### **Data Preprocessing: Feature & Tuple Duplication and Inconsistency**
#### **What is Feature Duplication ?**
Feature duplication occurs when a dataset contains redundant features that carry the same (or very similar) information. This adds noise and complexity without adding value to a model.

#### **When it occurs ?**
* During data collection
* Merging multiple data sources
* Engineering new features on top of existing ones

#### **Why is it a problem ?**
* Overfitting
* Slower training
* Misleading feature importance
* Multicollinearity


<p id="p1" style="font-weight:bolder;text-align:center;">Importing the data from a csv file</p>

In [2]:
import pandas as pd
import numpy as np
with open("data.csv", "r") as f:
    header = f.readline().strip().split(",")
df = pd.read_csv("data.csv")
df.columns = header

#importing data

<img src="csv.png" alt="" style="width:50%;height:100%;">

In [3]:
df

,Product_ID,Game_Title,Game_Name,Platform,Price,Price,Price_With_Tax,Metacritic_Score,User_Rating,Stock_Status,Publisher
0,501,Elden Ring,Elden Ring,PC,59.99,49.99,71.99,96,9.6,In Stock,FromSoftware
1,501,Elden Ring,Elden Ring,PC,59.99,49.99,71.99,97,9.7,In Stock,FromSoftware
2,502,Cyberpunk 2077,Cyberpunk 2077,PS5,29.99,19.99,35.99,85,8.4,In Stock,CD Projekt
3,502,Cyberpunk 2077,Cyberpunk 2077,PS5,49.99,39.99,59.99,89,9.1,In Stock,CD Projekt
4,503,Halo Infinite,Halo Infinite,Xbox,59.99,49.99,71.99,86,8.8,Out of Stock,Xbox Game Studios
5,503,Halo Infinite,Halo Infinite,Xbox,59.99,49.99,71.99,86,8.5,OUT OF STOCK,Xbox Game Studios
6,504,Stardew Valley,Stardew Valley,Switch,14.99,9.99,17.99,89,8.9,In Stock,ConcernedApe
7,504,Stardew Valley,Stardew Valley,Switch,14.99,9.99,17.99,85,8.3,Discontinued,ConcernedApe
8,505,Minecraft,Minecraft,PC,26.95,19.95,32.34,90,9.1,In Stock,Mojang
9,506,Terraria,Terraria,PC,9.99,4.99,11.99,81,7.7,In Stock,Re-Logic


<h1 style="text-align:center"><b>This is how to remove these duplicate features</b></h1>

### **Duplicate data can be split into different types like :**
* Features having the exact same names

In [4]:
df=df.loc[:,~df.columns.duplicated()]
df


,Product_ID,Game_Title,Game_Name,Platform,Price,Price_With_Tax,Metacritic_Score,User_Rating,Stock_Status,Publisher
0,501,Elden Ring,Elden Ring,PC,59.99,71.99,96,9.6,In Stock,FromSoftware
1,501,Elden Ring,Elden Ring,PC,59.99,71.99,97,9.7,In Stock,FromSoftware
2,502,Cyberpunk 2077,Cyberpunk 2077,PS5,29.99,35.99,85,8.4,In Stock,CD Projekt
3,502,Cyberpunk 2077,Cyberpunk 2077,PS5,49.99,59.99,89,9.1,In Stock,CD Projekt
4,503,Halo Infinite,Halo Infinite,Xbox,59.99,71.99,86,8.8,Out of Stock,Xbox Game Studios
5,503,Halo Infinite,Halo Infinite,Xbox,59.99,71.99,86,8.5,OUT OF STOCK,Xbox Game Studios
6,504,Stardew Valley,Stardew Valley,Switch,14.99,17.99,89,8.9,In Stock,ConcernedApe
7,504,Stardew Valley,Stardew Valley,Switch,14.99,17.99,85,8.3,Discontinued,ConcernedApe
8,505,Minecraft,Minecraft,PC,26.95,32.34,90,9.1,In Stock,Mojang
9,506,Terraria,Terraria,PC,9.99,11.99,81,7.7,In Stock,Re-Logic


* Features having the exact same values for each row (hard duplicate)

In [5]:
df=df.loc[:, ~df.T.duplicated()]
df

,Product_ID,Game_Title,Platform,Price,Price_With_Tax,Metacritic_Score,User_Rating,Stock_Status,Publisher
0,501,Elden Ring,PC,59.99,71.99,96,9.6,In Stock,FromSoftware
1,501,Elden Ring,PC,59.99,71.99,97,9.7,In Stock,FromSoftware
2,502,Cyberpunk 2077,PS5,29.99,35.99,85,8.4,In Stock,CD Projekt
3,502,Cyberpunk 2077,PS5,49.99,59.99,89,9.1,In Stock,CD Projekt
4,503,Halo Infinite,Xbox,59.99,71.99,86,8.8,Out of Stock,Xbox Game Studios
5,503,Halo Infinite,Xbox,59.99,71.99,86,8.5,OUT OF STOCK,Xbox Game Studios
6,504,Stardew Valley,Switch,14.99,17.99,89,8.9,In Stock,ConcernedApe
7,504,Stardew Valley,Switch,14.99,17.99,85,8.3,Discontinued,ConcernedApe
8,505,Minecraft,PC,26.95,32.34,90,9.1,In Stock,Mojang
9,506,Terraria,PC,9.99,11.99,81,7.7,In Stock,Re-Logic


* Features having  |corrolation|> r  (high corrolation)

In [6]:
df.select_dtypes(include=np.number).corr().abs()

,Product_ID,Price,Price_With_Tax,Metacritic_Score,User_Rating
Product_ID,1.000000,0.079921,0.079923,0.091744,0.110871
Price,0.079921,1.000000,1.000000,0.152920,0.262330
Price_With_Tax,0.079923,1.000000,1.000000,0.152915,0.262324
Metacritic_Score,0.091744,0.152920,0.152915,1.000000,0.874228
User_Rating,0.110871,0.262330,0.262324,0.874228,1.000000


In [7]:

r=0.8
numeric_df = df.select_dtypes(include=np.number)     # Keep only numeric columns
corr_matrix = numeric_df.corr().abs()  # Compute correlation matrix
to_drop = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if corr_matrix.iloc[i, j] > r:
            to_drop.add(corr_matrix.columns[i])
clean_numeric = numeric_df.drop(columns=list(to_drop))

df=pd.concat([clean_numeric, df.select_dtypes(exclude=np.number)], axis=1) # Combine back with non-numeric columns
df

,Product_ID,Price,Metacritic_Score,Game_Title,Platform,Stock_Status,Publisher
0,501,59.99,96,Elden Ring,PC,In Stock,FromSoftware
1,501,59.99,97,Elden Ring,PC,In Stock,FromSoftware
2,502,29.99,85,Cyberpunk 2077,PS5,In Stock,CD Projekt
3,502,49.99,89,Cyberpunk 2077,PS5,In Stock,CD Projekt
4,503,59.99,86,Halo Infinite,Xbox,Out of Stock,Xbox Game Studios
5,503,59.99,86,Halo Infinite,Xbox,OUT OF STOCK,Xbox Game Studios
6,504,14.99,89,Stardew Valley,Switch,In Stock,ConcernedApe
7,504,14.99,85,Stardew Valley,Switch,Discontinued,ConcernedApe
8,505,26.95,90,Minecraft,PC,In Stock,Mojang
9,506,9.99,81,Terraria,PC,In Stock,Re-Logic


#### **What is Tuple Duplication?**
Tuple duplication occurs when the exact same row appears more than once in a database table. This adds redundancy and clutter without contributing any new or useful information.

#### **When it occurs?**
* During manual data entry
* Merging multiple data sources
* Importing data from external systems without deduplication checks

#### **Why is it a problem?**
* Wastes storage space
* Slower queries and performance
* Unreliable record counts
#### **What is Tuple Inconsistency?**
Tuple inconsistency occurs when the same real-world fact is stored with conflicting values across different rows. The data exists more than once but with different and contradicting information.

#### **When it occurs?**
* When updates are applied to only some copies of a record
* During data migration or merging from different sources

#### **Why is it a problem?**
* Data cannot be trusted
* Applications display wrong information to users
* Update anomalies keep reintroducing errors
* Impossible to determine which version of the data is correct
* Makes normalization and cleanup significantly harder

### **Solution:**

##### *TUPLE DUPLICATION*

In [ ]:
# Removes rows that are exact clones across all columns
df = df.drop_duplicates()
df

##### *SYNTACTIC INCONSISTENCY*
In the context of data, a syntactic difference means two things look different on the surface but are actually the same in meaning. (e.g. `new_york` and `New_York`)

In [ ]:
# Standardizes 'Stock_Status' to Title Case
df['Stock_Status'] = df['Stock_Status'].str.title().str.strip()
# Standardizes publisher
df['Publisher'] = df['Publisher'].str.strip()
df

Here, we use string manipulation to apply standardization, so we can get rid of case sensitivity issues.

##### *SEMANTIC INCONSISTENCY*
Semantic inconsistency occurs when two pieces of data mean the same thing but are represented differently, causing the system to treat them as distinct when they shouldn't be.

In [ ]:
# Groups by the Key (Product_ID) and resolves conflicting values
# Note: Pandas renames the second 'Price' column to 'Price.1' automatically
# The as_index=False parameter is telling pandas: don't treat row numbers as actual data
df_cleaned = df.groupby('Product_ID', as_index=False).agg({
    'Game_Title': 'first',
    'Platform': 'first',
    'Price': 'max',            # Resolves price conflicts by picking the highest
    'Metacritic_Score': 'mean', # Averages conflicting scores
    'Stock_Status': 'first',    
    'Publisher': 'first'        
})
df_cleaned

##### *OUTPUT CLEANED DATA*

In [ ]:
# Save the results to a file
df_cleaned.to_csv('cleaned_data_1.csv', index=False)
df_cleaned.head()